In [3]:
import Pkg; Pkg.activate(@__DIR__); Pkg.add("ForwardDiff"); Pkg.instantiate()

  Activating project at `/workspaces/lecture-notebooks/Lecture 15`
   Resolving package versions...
    Updating `/workspaces/lecture-notebooks/Lecture 15/Project.toml`
  [f6369f11] + ForwardDiff v1.2.0
    Updating `/workspaces/lecture-notebooks/Lecture 15/Manifest.toml`
  [bbf7d656] + CommonSubexpressions v0.3.1
  [163ba53b] + DiffResults v1.1.0
  [b552c78f] + DiffRules v1.15.1
  [ffbed154] + DocStringExtensions v0.9.5
  [f6369f11] + ForwardDiff v1.2.0
  [92d709cd] + IrrationalConstants v0.2.4
  [692b3bcd] + JLLWrappers v1.7.1
  [2ab3a3ac] + LogExpFunctions v0.3.29
  [1914dd2f] + MacroTools v0.5.16
  [77ba4419] + NaNMath v1.1.3
  [21216c6a] + Preferences v1.5.0
  [276daf66] + SpecialFunctions v2.5.1
  [1e83bf80] + StaticArraysCore v1.4.3
  [efe28fd5] + OpenSpecFun_jll v0.5.6+0
  [56f22d72] + Artifacts
  [ade2ca70] + Dates
  [8f399da3] + Libdl
  [37e2e46d] + LinearAlgebra
  [de0858da] + Printf
  [9a3f8284] + Random
  [ea8e919c] + SHA v0.7.0
  [fa267f1f] + TOML v1.0.3
  [4ec0a83e] + Un

In [4]:
using LinearAlgebra
using ForwardDiff

In [5]:
function hat(v)
    return [0 -v[3] v[2];
            v[3] 0 -v[1];
            -v[2] v[1] 0]
end

hat (generic function with 1 method)

In [6]:
function L(q)
    s = q[1]
    v = q[2:4]
    L = [s    -v';
         v  s*I+hat(v)]
    return L
end

L (generic function with 1 method)

In [7]:
function R(q)
    s = q[1]
    v = q[2:4]
    R = [s    -v';
         v  s*I-hat(v)]
    return R
end

R (generic function with 1 method)

In [8]:
T = Diagonal([1; -ones(3)])
H = [zeros(1,3); I];

In [10]:
function G(q)
    G = L(q)*H
end

function Q(q)
    return H'*(R(q)'*L(q))*H
end

Q (generic function with 1 method)

In [11]:
#Generate a random quaternion
qtrue = randn(4)
qtrue = qtrue/norm(qtrue)

Qtrue = Q(qtrue) #Generate equivalent rotation matrix

3×3 Matrix{Float64}:
  0.973592  -0.147427   0.174309
  0.191994   0.941854  -0.275772
 -0.123518   0.301956   0.945286

In [12]:
#Generate data
vN = randn(3,10) #Generate some random world-frame vectors

#normalize
for k = 1:10
    vN[:,k] .= vN[:,k]./norm(vN[:,k])
end

vB = Qtrue'*vN #generate body-frame vectors

3×10 Matrix{Float64}:
  0.618631   0.521152   0.704627   …   0.877332  -0.754074  -0.863628
 -0.753995  -0.818716  -0.0679442      0.372681  -0.451973  -0.451608
  0.22088   -0.241049  -0.706318      -0.30232   -0.476542   0.224047

In [13]:
function residual(q)
    r = vN - Q(q)*vB
    return r[:]
end

residual (generic function with 1 method)

In [14]:
#Random initial guess
q = randn(4)
q = q/norm(q)

4-element Vector{Float64}:
  0.5714963042986213
  0.3101275267462211
 -0.687619642831526
 -0.32309769129403315

In [16]:
#Gauss-Newton Method
ϕ = ones(3)
iter = 0
while maximum(abs.(ϕ)) > 1e-8
    r = residual(q)
    dr = ForwardDiff.jacobian(residual, q)
    ∇r = dr*G(q)
    ϕ = -(∇r'*∇r)\(∇r'*r) #3-parameter update computed with gauss-newton
    q = L(q)*[sqrt(1-ϕ'*ϕ); ϕ] #multiplicative update applied to q
    iter += 1
end

In [17]:
q-qtrue

4-element Vector{Float64}:
 1.9648745522780922
 0.2940276109398381
 0.15157562107539518
 0.17274428763518176

In [18]:
q+qtrue

4-element Vector{Float64}:
  1.1102230246251565e-16
  2.7755575615628914e-17
 -1.3877787807814457e-17
  0.0